<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/2_0_Computing_variance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

$\frac{\sum \left(x_i - \bar{x}\right)^2}{N}$

In [1]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.03-Extending-Algorithms/Sources/ach.h -nv -O Sources/ach.h

2026-09-08 01:33:15 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.03-Extending-Algorithms/Sources/ach.h [2787/2787] -> "Sources/ach.h" [1]


In [12]:
%%writefile Sources/variance.cpp
#include "ach.h"

float mean(thrust::universal_vector<float> &vec)
{
  return thrust::reduce(thrust::device, vec.begin(), vec.end(), 0.0f, thrust::plus<float>())/vec.size();
}


float variance(thrust::universal_vector<float> &vec, float mean)
{

  auto squared_diff = thrust::make_transform_iterator(vec.begin(),
                                                      [mean]__host__ __device__ (float value){
                                                        return (value - mean)*(value - mean);
                                                      });

  return thrust::reduce(thrust::device, squared_diff, squared_diff + vec.size())/vec.size();
}


int main()
{
  float ambient_temp = 20;
  thrust::universal_vector<float> prev{42, 24, 50};
  thrust::universal_vector<float> next{0, 0, 0};

  std::printf("step     variance\n");
  for(int step = 0; step < 3; step ++)
  {
    thrust::transform(thrust::device, prev.begin(), prev.end(), next.begin(),
    [=] __host__ __device__ (float temp){

      return temp + 0.5*(ambient_temp - temp);
    });

    std::printf("%d        %.2f\n", step, variance(next, mean(next)));

    next.swap(prev);
  }
}


Overwriting Sources/variance.cpp


In [13]:
!nvcc --extended-lambda -o /tmp/a.out Sources/variance.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

step     variance
0        29.56
1        7.39
2        1.85
